In [5]:
import pandas as pd
import requests
import time
import os

def get_repo_info(repo, token):
    headers = {
        'Authorization': f'token {token}'
    }
    
    base_url = f'https://api.github.com/repos/{repo}'
    
    # Get basic repo information
    repo_info = requests.get(base_url, headers=headers).json()
    
    # Get contributors count
    contributors_url = f'{base_url}/contributors'
    contributors = requests.get(contributors_url, headers=headers).json()
    contributors_count = len(contributors)
    
    # Get open issues count
    issues_url = f'{base_url}/issues?state=open'
    open_issues = requests.get(issues_url, headers=headers).json()
    open_issues_count = len(open_issues)
    
    # Get closed issues count
    issues_url = f'{base_url}/issues?state=closed'
    closed_issues = requests.get(issues_url, headers=headers).json()
    closed_issues_count = len(closed_issues)
    
    # Get open pull requests count
    pulls_url = f'{base_url}/pulls?state=open'
    open_pulls = requests.get(pulls_url, headers=headers).json()
    open_pulls_count = len(open_pulls)
    
    # Get closed pull requests count
    pulls_url = f'{base_url}/pulls?state=closed'
    closed_pulls = requests.get(pulls_url, headers=headers).json()
    closed_pulls_count = len(closed_pulls)
    
    # Get releases count
    releases_url = f'{base_url}/releases'
    releases = requests.get(releases_url, headers=headers).json()
    releases_count = len(releases)
    
    # Get total commits count
    commits_url = f'{base_url}/commits'
    commits = requests.get(commits_url, headers=headers).json()
    commits_count = len(commits)
    
    # Get other repository information
    forks_count = repo_info.get('forks_count', 0)
    stargazers_count = repo_info.get('stargazers_count', 0)
    watchers_count = repo_info.get('watchers_count', 0)
    
    # Compile all information into a dictionary
    info = {
        'CreationDate': repo_info.get('created_at'),
        'Language': repo_info.get('language'),
        'Contributors': contributors_count,
        'OpenIssues': open_issues_count,
        'ClosedIssues': closed_issues_count,
        'Commits': commits_count,
        'OpenPullRequest': open_pulls_count,
        'ClosedPullRequest': closed_pulls_count,
        'Releases': releases_count,
        'Forks': forks_count,
        'Stars': stargazers_count,
        'Watchers': watchers_count
    }
    
    return info

token = os.environ['GITHUB_TOKEN']
df = pd.read_csv('CycloneNSpdxTools.csv')

for index, row in df.iterrows():
    info = get_repo_info(row['Repo'], token)
    for key, value in info.items():
        df.at[index, key] = value
    df.at[index, 'Format'] = row['Format']
    df.at[index, 'Tool'] = row['Tool']
    df.at[index, 'Repo'] = row['Repo']
    time.sleep(1)

df.to_csv('CycloneNSpdxTools.csv', index=False)

In [15]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pickle
from collections import Counter

# Load data from files
with open('commit_counts_by_repo.pkl', 'rb') as f:
    commit_count_repos = pickle.load(f)

df = pd.read_csv('CycloneNSpdxTools.csv')

# Initialize counters
spdx_counter = Counter()
cyclonedx_counter = Counter()

# Update counters based on the dataframe
for index, row in df.iterrows():
    if row['Format'] == 'SPDX':
        spdx_counter.update(v for v in commit_count_repos[row['Repo']].values() if v > 1)
    elif row['Format'] == 'CycloneDx':
        cyclonedx_counter.update(v for v in commit_count_repos[row['Repo']].values() if v > 1)

# Prepare data for plotting
spdx_data = [count for count in spdx_counter.elements()]
cyclonedx_data = [count for count in cyclonedx_counter.elements()]

# Create the violin plot
fig = go.Figure()

fig.add_trace(go.Violin(
    y=spdx_data,
    x=['SPDX'] * len(spdx_data),
    box_visible=True,
    line_color='blue',
    spanmode='hard',
    name='SPDX'
))

fig.add_trace(go.Violin(
    y=cyclonedx_data,
    x=['CycloneDx'] * len(cyclonedx_data),
    box_visible=True,
    line_color='green',
    spanmode='hard',
    name='CycloneDx'
))

# Update layout for log scale
fig.update_layout(
    title="Combined Violin Plot of Commit Counts",
    xaxis_title="Format",
    yaxis_title="Commit Counts",
    yaxis_type="log",
    violingap=0.4,  # Adjusts the gap between violins
    violinmode='group'  # Groups the violins side by side
)

# Show the plot
pio.show(fig)

In [14]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pickle
from collections import Counter

# Load data from files
with open('commit_counts_by_repo.pkl', 'rb') as f:
    commit_count_repos = pickle.load(f)

df = pd.read_csv('CycloneNSpdxTools.csv')

# Initialize counters
spdx_counter = Counter()
cyclonedx_counter = Counter()

# Update counters based on the dataframe
for index, row in df.iterrows():
    if row['Format'] == 'SPDX':
        spdx_counter.update(v for v in commit_count_repos[row['Repo']].values() if v > 1)
    elif row['Format'] == 'CycloneDx':
        cyclonedx_counter.update(v for v in commit_count_repos[row['Repo']].values() if v > 1)

# Prepare data for plotting
spdx_data = [count for count in spdx_counter.elements()]
cyclonedx_data = [count for count in cyclonedx_counter.elements()]

# Create the unified violin plot
fig = go.Figure()

fig.add_trace(go.Violin(
    y=spdx_data,
    x=['Commit Counts'] * len(spdx_data),
    box_visible=True,
    line_color='blue',
    spanmode='hard',
    side='negative',  # Set to show on the left side
    name='SPDX'
))

fig.add_trace(go.Violin(
    y=cyclonedx_data,
    x=['Commit Counts'] * len(cyclonedx_data),
    box_visible=True,
    line_color='green',
    spanmode='hard',
    side='positive',  # Set to show on the right side
    name='CycloneDx'
))

# Update layout for log scale
fig.update_layout(
    title="Unified Violin Plot of Commit Counts",
    xaxis_title="Format",
    yaxis_title="Commit Counts",
    yaxis_type="log",
    violingap=0.4,  # Adjusts the gap between violins
    violinmode='overlay',  # Overlays the violins side by side
)

# Show the plot
pio.show(fig)
